In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0,"..")
from src.fsm.name_matcher import filter_names
from src.models.load_init import load_vocab_inverse
from llm_sdk import Small_LLM_Model

from src.models.load_init import load_function_catalog
from src.vocab.filtering import build_plausible_vocab
from src.fsm.name_matcher import allowed_token_ids_for_token 
from src.fsm.name_matcher import filter_names
from src.generation.decoding import mask_logits, select_next_token
model = Small_LLM_Model()
path_catalog = "../data/input/functions_definition.json"
catalog = load_function_catalog(path_catalog)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
vocab_inverse = load_vocab_inverse(model.get_path_to_vocab_file())
allowed_chars = {c for i in catalog.functions for c in i.name}
plausible_vocab = build_plausible_vocab(vocab_inverse, allowed_chars)
noms_valides = [i.name for i in catalog.functions]

In [ ]:
def name_matcher(full_prefix: str,
                 noms_valides: list,
                 plausible_vocab: dict,
                 model: Small_LLM_Model) -> str:
    """
    Generate a name that matches the allowed names in the catalog, given a full prefix.

    Args:
        full_prefix (str): The prefix to start the name generation.
        noms_valides (list): A list of valid names to match against.
        plausible_vocab (dict): A dictionary mapping token IDs to their corresponding text.
        model (Small_LLM_Model): The language model used for generating names.
    Returns:
        str: A generated name that matches the allowed names in the catalog.
    """
    partial_text = ""
    while True:
        allowed_token_ids = allowed_token_ids_for_token(partial_text, noms_valides, plausible_vocab)
        
        if not allowed_token_ids:
            break
        full_text = full_prefix + partial_text
        input_ids = model.encode(full_text).tolist()[0]
        logits = model.get_logits_from_input_ids(input_ids)
        masked_logits = mask_logits(allowed_token_ids, logits)
        best_id = select_next_token(masked_logits)
        
        token_text = plausible_vocab.get(best_id,"")
        partial_text += (token_text or "")
    
    return full_prefix + partial_text + '"'